In [1]:
# ==========================================================
# FEATURE_SELECTION.PY
# ==========================================================

# ==========================================================
# IMPORTS
# ==========================================================
import pandas as pd
import numpy as np

from sklearn.feature_selection import (
    mutual_info_classif,
    SelectKBest,
    f_classif
)

# ==========================================================
# LOAD DATA
# ==========================================================
df = pd.read_csv("data_processed.csv")

# ==========================================================
# TARGET
# ==========================================================
target = "fraud_bool"

X = df.drop(columns=[target])

y = df[target]

# ==========================================================
# ANOVA F-SCORE
# ==========================================================

anova_selector = SelectKBest(
    score_func=f_classif,
    k=15
)

anova_selector.fit(X, y)

anova_scores = pd.DataFrame({
    "feature": X.columns,
    "anova_score": anova_selector.scores_
})

anova_scores = anova_scores.sort_values(
    by="anova_score",
    ascending=False
)

print("\n--- ANOVA SCORES ---")
print(anova_scores)

# ==========================================================
# MUTUAL INFORMATION
# ==========================================================

mi_scores = mutual_info_classif(X, y)

mi_df = pd.DataFrame({
    "feature": X.columns,
    "mi_score": mi_scores
})

mi_df = mi_df.sort_values(
    by="mi_score",
    ascending=False
)

print("\n--- MUTUAL INFORMATION ---")
print(mi_df)

# ==========================================================
# CORRELATION FILTER
# ==========================================================

corr_matrix = X.corr().abs()

upper = corr_matrix.where(
    np.triu(
        np.ones(corr_matrix.shape),
        k=1
    ).astype(bool)
)

to_drop = [
    column for column in upper.columns
    if any(upper[column] > 0.90)
]

print("\nHighly Correlated Features:")
print(to_drop)

# ==========================================================
# FINAL FEATURE SET
# ==========================================================

top_features = (
    mi_df["feature"]
    .head(20)
    .tolist()
)

print("\nSelected Features:")
print(top_features)

# ==========================================================
# KEEP TEMPORAL FEATURE
# ==========================================================

final_features = top_features.copy()

if "month" not in final_features:
    final_features.append("month")

# ==========================================================
# CREATE FINAL DATASET
# ==========================================================

selected_df = df[
    final_features + [target]
]

# ==========================================================
# SAVE DATASET
# ==========================================================

selected_df.to_csv(
    "data_processed_alternative.csv",
    index=False
)

print("\n========================================")
print("FEATURE SELECTION COMPLETED")
print("========================================")

print(f"Number of selected features: {len(final_features)}")

print("\nFinal Features:")
print(final_features)

print("\nDataset saved as: data_selected.csv")


--- ANOVA SCORES ---
                             feature  anova_score
17                    housing_status  6121.208170
15                 credit_risk_score  5012.689169
22             proposed_credit_limit  4770.766210
4                       customer_age  3979.557852
26                         device_os  3356.022639
27                keep_alive_session  2536.139830
0                             income  2036.265870
13  date_of_birth_distinct_emails_4w  1871.782022
1              name_email_similarity  1350.149408
28         device_distinct_emails_8w  1336.632204
21                   has_other_cards  1237.497262
18                  phone_home_valid  1235.485454
3       current_address_months_count  1128.122874
7                       payment_type   801.778137
16                     email_is_free   771.077387
6             intended_balcon_amount   601.809238
14                 employment_status   470.083441
9                        velocity_6h   285.432634
23                   foreign